# Laya × Memory Fusion — All Full and Sliding Attention Layers

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/codex/laya-memoryfusion-agreement/notebooks/Laya_MemoryFusion_Colab.ipynb)

Uses **`convaiinnovations/laya`** as the unchanged teacher. Attempts every ModernBERT full and sliding attention layer in encoder order.

- Full layers use bidirectional feature memory, sparse local mixing, and direct V.
- Sliding layers keep the original window radius in both mixing branches.
- Cached block fitting avoids a complete model forward on every optimizer step.
- Recurrent GDN2 is opt-in for full layers; disabled by default to avoid Python token loops.
- Best-probe restoration includes the memory activation flag. Decision refinement retains the best agreement checkpoint and excludes functional probe examples from its training batches.
- Each replacement must pass local fidelity **and cumulative student** decision gates. Failed layers retain original attention.
- `complete` means every attention layer was replaced; `partial` lists remaining original layers.

This is approximate distillation, not an exact mathematical substitution. Full conversion and faster inference are measured outcomes, not guarantees. Sparse softmax remains inside the replacement's local branch.


## 1. Setup
A T4/L4/A100 runtime is recommended. The setup pulls the latest TinyCeNN-LM and Laya source, then performs a syntax preflight before training.


In [1]:
import os, sys, subprocess, pathlib, importlib, compileall
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if pathlib.Path("/content").exists():
    WORK = pathlib.Path("/content")
elif pathlib.Path("/kaggle/working").exists():
    WORK = pathlib.Path("/kaggle/working")
else:
    WORK = pathlib.Path.cwd()

REVISION = "codex/laya-memoryfusion-agreement"
REPO = WORK / "TinyCeNN-LM-MemoryFusion"
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "-q", "--branch", REVISION, "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)])
else:
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REVISION])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", REVISION])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REVISION])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/NandhaKishorM/laya.git",
    "datasets", "pandas", "pyarrow", "safetensors"
])

SRC = str(REPO / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
# Reload this package after updates in a previously used runtime.
for name in list(sys.modules):
    if name == "tinycenn_lm" or name.startswith("tinycenn_lm."):
        del sys.modules[name]
importlib.invalidate_caches()

LAB_SRC = REPO / "src" / "tinycenn_lm" / "laya_lab"
assert compileall.compile_dir(str(LAB_SRC), quiet=1), "Python syntax preflight failed in tinycenn_lm/laya_lab"

import torch, json, pandas as pd
from tinycenn_lm.laya_lab.memory_fusion_v3 import (
    LayaMemoryFusionV3Config,
    run_memory_fusion_v3,
)

print("repo:", REPO)
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


repo: /content/TinyCeNN-LM-MemoryFusion
torch: 2.11.0+cu130
GPU: Tesla T4


## 2. All-attention configuration

`target_all_attention=True` discovers every full and sliding layer. For a quick single-layer test, set it to `False` and use `candidate_layer=12`; an explicit subset can use `target_layers=(0, 1, 2)` with all-layer mode off.

The router/gain learning rate starts at 0.01 after warm-up. Feature maps use 0.001 and copied output projections use 0.00005. Plateau cuts and best-probe restoration control regression. These are starting settings; this revision has not been trained on a T4 yet. Local fidelity, KL and accuracy thresholds are unchanged; teacher agreement now targets 100% on the validation gate. This is a target, not a guaranteed result on unseen data. Decision refinement uses a stronger KL loss plus a teacher-margin loss, and restores its best checkpoint even if further fitting is needed.


In [4]:
MODEL_ID = "convaiinnovations/laya"

cfg = LayaMemoryFusionV3Config(
    model_id=MODEL_ID,
    seed=2026,
    output_dir=str(WORK / "laya_tinycenn"),

    target_all_attention=True,
    target_layers=None,
    candidate_layer=12,       # used only when all-layer mode is off
    enable_recurrent_memory=False,  # opt-in, full layers only

    feature_dim=64,
    memory_rank=64,
    dilations=(1, 2, 4, 8, 16, 32, 64),

    train_cases=480,
    train_max_len=512,
    batch_size=4,
    train_cache_batches=48,   # 192 cached training examples
    probe_cache_batches=8,    # fixed 32-example probe
    cache_on_device=True,      # CPU fallback when cache exceeds GPU headroom

    # Turbo schedule: fewer steps, more useful movement per step.
    functional_steps=320,
    max_rounds=3,
    check_every=20,
    min_steps_before_check=40,
    fast_learning_rate=1.0e-2,   # fusion/router/gains
    core_learning_rate=1.0e-3,   # Hedgehog + GDN2 feature maps
    output_learning_rate=5.0e-5, # copied Laya Wo
    warmup_steps=10,
    round_lr_decay=0.70,
    weight_decay=2e-4,

    # Push direction/cosine much harder once reconstruction is reasonable.
    cosine_weight=0.30,
    near_gate_cosine_weight=0.85,
    near_gate_nmse=0.38,

    # Used only when enable_recurrent_memory=True.
    memory_enable_nmse=0.36,
    memory_enable_cosine=0.80,

    # Keep fidelity/accuracy gates; require exact teacher gate agreement.
    max_local_nmse=0.20,
    min_local_cosine=0.90,
    min_teacher_agreement=0.9,  # exact gate agreement; 0.995 allows near-perfect
    max_mean_kl=0.05,
    max_accuracy_drop=0.02,

    gate_cases=80,
    final_cases=160,

    decision_refine_steps=120,
    decision_refine_lr_scale=0.10,  # stable 1e-4 decision refinement
    decision_kl_weight=1.0,
    decision_local_weight=0.25,
    decision_margin_weight=0.50,
    decision_margin=0.02,
)
print(cfg)


LayaMemoryFusionV3Config(model_id='convaiinnovations/laya', seed=2026, output_dir='/content/laya_tinycenn', candidate_layer=12, target_all_attention=True, target_layers=None, enable_recurrent_memory=False, feature_dim=64, memory_rank=64, dilations=(1, 2, 4, 8, 16, 32, 64), train_cases=480, train_max_len=512, batch_size=4, train_cache_batches=48, probe_cache_batches=8, functional_steps=320, max_rounds=3, check_every=20, min_steps_before_check=40, fast_learning_rate=0.01, core_learning_rate=0.001, output_learning_rate=5e-05, warmup_steps=10, round_lr_decay=0.7, weight_decay=0.0002, cosine_weight=0.3, near_gate_cosine_weight=0.85, near_gate_nmse=0.38, memory_enable_nmse=0.36, memory_enable_cosine=0.8, max_local_nmse=0.2, min_local_cosine=0.9, min_teacher_agreement=0.9, max_mean_kl=0.05, max_accuracy_drop=0.02, gate_cases=80, final_cases=160, decision_refine_steps=120, decision_refine_lr_scale=0.1, decision_kl_weight=1.0, action_kl_weight=0.01, distill_temperature=1.0, cache_on_device=True

## 3. Fit all layers and validate the cumulative student

Teacher I/O is cached per layer and released before the next candidate. Local fitting restores the best fixed-probe checkpoint, then decision gates evaluate the student with every previously accepted replacement. Earlier accepted layers stay frozen. Gate examples come from held-out train rows; final evaluation uses the official test split.

Every candidate saves progress. All-layer training takes longer than a single-layer experiment; early stopping avoids spending the full budget on layers that already pass.


In [5]:
teacher, student, report = run_memory_fusion_v3(cfg)


Loading Laya teacher: convaiinnovations/laya


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/agent.py:785: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


Teacher gate accuracy: 0.3575
Gate workflows: {'agent_trace_observability': 100, 'customer_service': 100, 'invoice_processing': 100, 'security_incidents': 100}
Candidate layers: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]

=== Candidate layer 0: full_attention ===
Building train teacher cache: 48 batches × 4
  cached 1/48
  cached 8/48
  cached 16/48
  cached 24/48
  cached 32/48
  cached 40/48
  cached 48/48
train cache ready in 6.7s
Building probe teacher cache: 8 batches × 4
  cached 1/8
  cached 8/8
probe cache ready in 1.1s
Teacher I/O cache moved to GPU for zero-copy functional fitting.

=== V3 layer 0: functional round 1/3 ===
round 1 start probe: NMSE=2.6274 cos=0.4394
round=1 step=001/320 nmse=2.7420 cos=0.4043 cw=0.30 lr=1.0e-03 grad=3.396
round=1 step=010/320 nmse=1.1275 cos=0.4905 cw=0.30 lr=1.0e-02 grad=1.412
round=1 step=020/320 nmse=0.6905 cos=0.5762 cw=0.30 lr=1.0e-02 grad=0.697
round=1 step=030/320 nmse=0.4713 

## 4. Result summary
If no layer passes, the final student is restored to original Laya attention and the report explicitly says 'failed_no_accepted_layers'. Identical teacher/student metrics are therefore never presented as a successful conversion.


In [6]:
summary = pd.DataFrame([
    {
        "model": "Laya teacher",
        **{k: report["teacher_final"].get(k) for k in [
            "accuracy", "soft_accuracy", "brier", "brier_vs_soft",
            "ece", "score_mae", "ms_per_case"
        ]},
    },
    {
        "model": report["architecture"],
        **{k: report["student_final"].get(k) for k in [
            "accuracy", "soft_accuracy", "brier", "brier_vs_soft",
            "ece", "score_mae", "ms_per_case"
        ]},
        "teacher_agreement": report["student_final"].get("teacher_agreement"),
        "teacher_KL": report["student_final"].get("mean_teacher_kl"),
    },
])
display(summary)

print("Status:", report["status"])
print("Accepted attention layers:", report["accepted_layers"])
print("All attention replaced:", report["all_attention_replaced"])
print("Remaining original layers:", report["remaining_attention_layers"])
print("Replacement parameters:", f'{report["replacement_parameters"]:,}')
print("Measured demo speedup:", round(report["latency"]["speedup"], 3), "x")
print("Gate/final disjoint:", report["gate_final_disjoint"])
print("Latency:", json.dumps(report["latency"], indent=2))


,model,accuracy,soft_accuracy,brier,brier_vs_soft,ece,score_mae,ms_per_case,teacher_agreement,teacher_KL
0,Laya teacher,0.32750,0.327847,0.762807,0.316246,0.205094,0.680353,133.762375,NaN,NaN
1,memory_fusion_v3,0.32125,0.326645,0.749784,0.301371,0.191019,0.696259,435.143949,0.8775,0.023025


Status: partial
Accepted attention layers: [0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 17, 20, 22, 23, 24, 25, 26, 27]
All attention replaced: False
Remaining original layers: [8, 11, 12, 13, 14, 15, 16, 18, 19, 21]
Replacement parameters: 87,623,424
Measured demo speedup: 0.229 x
Gate/final disjoint: True
Latency: {
  "teacher": {
    "median_ms": 40.82122649970188,
    "p95_ms": 42.813847199659,
    "mean_ms": 41.456727050126574,
    "repeats": 20
  },
  "student": {
    "median_ms": 178.27644649969443,
    "p95_ms": 185.26998240058674,
    "mean_ms": 177.65941579987157,
    "repeats": 20
  },
  "speedup": 0.2289771156043984
}


## 5. Layer-by-layer diagnostics

Fixed **PROBE** values determine local checkpoint selection. Decision-gate rows show cumulative teacher agreement, KL, accuracy and acceptance. A good local probe alone does not guarantee acceptance: earlier replacements can change downstream inputs. Rejected candidates keep their original attention.


In [7]:
rows = []
for h in report["history"]:
    gate = h.get("gate") or {}
    local = h.get("local") or {}
    rows.append({
        "layer": h.get("layer"),
        "round": h.get("round"),
        "stage": h.get("stage"),
        "local_pass": h.get("local_pass"),
        "accepted": h.get("accepted"),
        "best_step": local.get("step"),
        "nmse": local.get("nmse"),
        "cosine": local.get("cosine"),
        "teacher_agreement": gate.get("teacher_agreement"),
        "mean_teacher_kl": gate.get("mean_teacher_kl"),
        "accuracy": gate.get("accuracy"),
        "accuracy_drop": h.get("accuracy_drop"),
    })
display(pd.DataFrame(rows))


,layer,round,stage,local_pass,accepted,best_step,nmse,cosine,teacher_agreement,mean_teacher_kl,accuracy,accuracy_drop
0,0,1,functional,False,None,320.0,0.163283,0.898335,NaN,NaN,NaN,NaN
1,0,2,functional,True,None,320.0,0.128817,0.920918,NaN,NaN,NaN,NaN
2,0,2,decision_gate,None,True,320.0,0.128817,0.920918,0.9475,0.005673,0.3650,-0.0075
3,1,1,functional,True,None,160.0,0.144960,0.904817,NaN,NaN,NaN,NaN
4,1,1,decision_gate,None,False,160.0,0.144960,0.904817,0.8950,0.016535,0.3425,0.0150
...,...,...,...,...,...,...,...,...,...,...,...,...
98,26,1,functional,True,None,80.0,0.189179,0.909096,NaN,NaN,NaN,NaN
99,26,1,decision_gate,None,True,80.0,0.189179,0.909096,0.9000,0.024514,0.3650,-0.0075
100,27,1,functional,True,None,60.0,0.184011,0.904551,NaN,NaN,NaN,NaN
101,27,1,decision_gate,None,False,60.0,0.184011,0.904551,0.8975,NaN,0.3625,-0.0050


## 6. Laya Router smoke test

The adapted Agent keeps Laya's public API. The example below attaches the already-loaded student without loading a second model.


In [8]:
from laya import Router

state = {
    "from": "user@acme.com",
    "subject": "Duplicate charge on invoice #4411",
    "body": "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan.",
}
questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this request?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, system errors",
            "sales": "pricing, new contracts",
            "other": "everything else",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this request?",
        "criteria": ["not urgent", "soon", "critical deadline or blocking issue"],
    },
    "churn_risk": {
        "type": "noul",
        "instructions": "Does the user threaten to cancel or leave?",
    },
    "refund_requested": {
        "type": "noul",
        "instructions": "Does the user explicitly request a refund?",
    },
}

router = Router(preload=False)
router.attach("english", student)
res = router.predict(state, questions, model="english")

print("Department       :", res["answers"]["department"]["choice"])
print("Urgency score    :", res["answers"]["urgency"]["score"])
print("Churn risk       :", res["answers"]["churn_risk"]["noul"])
print("Refund requested :", res["answers"]["refund_requested"]["noul"])
print("Routing          :", res["routing"]["model"])
print("\nTeacher/student raw comparison:")
print(json.dumps(report["demo"], indent=2, ensure_ascii=False))


Department       : billing
Urgency score    : 1.5821
Churn risk       : 0.8175
Refund requested : 0.8173
Routing          : english

Teacher/student raw comparison:
{
  "teacher": {
    "model": "laya-rl-agent",
    "answers": {
      "department": {
        "type": "choice",
        "choice": "billing",
        "probabilities": {
          "billing": 0.9652,
          "technical": 0.014,
          "sales": 0.0101,
          "other": 0.0107
        },
        "confidence": 0.8639,
        "action": {
          "act_probability": 1.0
        }
      },
      "urgency": {
        "type": "score",
        "score": 1.4382,
        "legend": {
          "0": "not urgent",
          "1": "soon",
          "2": "critical deadline or blocking issue"
        },
        "probabilities": {
          "0": 0.1169,
          "1": 0.328,
          "2": 0.5551
        },
        "confidence": 0.1414,
        "action": {
          "act_probability": 1.0
        }
      },
      "churn_risk": {
        

## 7. Saved outputs

V3 writes its adapter, report, and best checkpoint for each functional round under `/content/laya_tinycenn/memory_fusion_v3/`.


In [9]:
from pathlib import Path
out = Path(cfg.output_dir) / "memory_fusion_v3"
print("Adapter:", out / "adapter.pt")
print("Report :", out / "report.json")
print("Round checkpoints:")
for p in sorted(out.glob("layer_*_round_*.pt")):
    print(" -", p.name)
print("\nReport preview:")
print((out / "report.json").read_text()[:5000])


Adapter: /content/laya_tinycenn/memory_fusion_v3/adapter.pt
Report : /content/laya_tinycenn/memory_fusion_v3/report.json
Round checkpoints:
 - layer_0_round_1.pt
 - layer_0_round_2.pt
 - layer_10_round_1.pt
 - layer_10_round_2.pt
 - layer_11_round_1.pt
 - layer_11_round_2.pt
 - layer_11_round_3.pt
 - layer_12_round_1.pt
 - layer_12_round_2.pt
 - layer_12_round_3.pt
 - layer_13_round_1.pt
 - layer_13_round_2.pt
 - layer_13_round_3.pt
 - layer_14_round_1.pt
 - layer_14_round_2.pt
 - layer_14_round_3.pt
 - layer_15_round_1.pt
 - layer_15_round_2.pt
 - layer_15_round_3.pt
 - layer_16_round_1.pt
 - layer_16_round_2.pt
 - layer_16_round_3.pt
 - layer_17_round_1.pt
 - layer_17_round_2.pt
 - layer_17_round_3.pt
 - layer_18_round_1.pt
 - layer_18_round_2.pt
 - layer_18_round_3.pt
 - layer_19_round_1.pt
 - layer_19_round_2.pt
 - layer_19_round_3.pt
 - layer_1_round_1.pt
 - layer_20_round_1.pt
 - layer_20_round_2.pt
 - layer_21_round_1.pt
 - layer_21_round_2.pt
 - layer_21_round_3.pt
 - layer_22_

## 8. Save accepted MemoryFusion V3 model to Hugging Face Hub\n\nPublishes only after at least one layer passes the strict gates. The Hub package stores the TinyCeNN adapter plus a loader that reconstructs the accepted MemoryFusion V3 layers on top of the declared base Laya model.\n

In [10]:
# === SAVE MEMORYFUSION V3 MODEL TO HUGGING FACE HUB ===
# Uploads a reloadable TinyCeNN package:
# base Laya + MemoryFusionV3 adapter.pt + report.json + loader.py.
from pathlib import Path
import json, os, shutil, subprocess
import torch
from huggingface_hub import HfApi, login

HF_REPO_ID = "vtava/Laya-MemoryFusion-V3" #@param {type:"string"}
HF_PRIVATE = False #@param {type:"boolean"}

accepted_layers = [int(i) for i in report.get("accepted_layers", [])]
if not accepted_layers:
    raise RuntimeError(
        "No MemoryFusionV3 layer passed the strict gates. "
        "HF upload is intentionally blocked so unchanged Laya is not published "
        "as a converted model."
    )

out = Path(cfg.output_dir) / "memory_fusion_v3"
adapter_path = out / "adapter.pt"
report_path = out / "report.json"

if not adapter_path.exists() or not report_path.exists():
    raise FileNotFoundError(
        "Run the training/evaluation cells first; "
        "memory_fusion_v3/adapter.pt and report.json are missing."
    )

# Verify that adapter.pt really contains exactly the accepted converted layers.
payload = torch.load(adapter_path, map_location="cpu", weights_only=False)
if payload.get("format") != "tinycenn-laya-attention-lab-v1":
    raise RuntimeError(f"Unexpected adapter format: {payload.get('format')!r}")

stored_layers = sorted(int(k) for k in payload.get("adapters", {}).keys())
if stored_layers != sorted(accepted_layers):
    raise RuntimeError(
        "adapter.pt/report mismatch: "
        f"report accepted_layers={sorted(accepted_layers)}, "
        f"adapter layers={stored_layers}"
    )

# Pin the TinyCeNN source revision used by this notebook when possible.
try:
    tinycenn_revision = subprocess.check_output(
        ["git", "-C", str(REPO), "rev-parse", "HEAD"],
        text=True,
    ).strip()
except Exception:
    tinycenn_revision = "codex/laya-memoryfusion-agreement"

export_dir = out / "hf_export"
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(adapter_path, export_dir / "adapter.pt")
shutil.copy2(report_path, export_dir / "report.json")

meta = {
    "format": payload["format"],
    "architecture": report.get("architecture", "memory_fusion_v3"),
    "base_model": cfg.model_id,
    "status": report.get("status"),
    "accepted_layers": accepted_layers,
    "candidate_layers": report.get("candidate_layers", []),
    "remaining_attention_layers": report.get("remaining_attention_layers", []),
    "all_attention_replaced": report.get("all_attention_replaced", False),
    "all_targets_replaced": report.get("all_targets_replaced", False),
    "feature_dim": cfg.feature_dim,
    "memory_rank": cfg.memory_rank,
    "dilations": list(cfg.dilations),
    "enable_recurrent_memory": cfg.enable_recurrent_memory,
    "fast_learning_rate": cfg.fast_learning_rate,
    "core_learning_rate": cfg.core_learning_rate,
    "output_learning_rate": cfg.output_learning_rate,
    "functional_steps": cfg.functional_steps,
    "max_rounds": cfg.max_rounds,
    "replacement_parameters": report.get("replacement_parameters"),
    "measured_speedup": report.get("latency", {}).get("speedup"),
    "tinycenn_repo": "https://github.com/vtavakkoli/TinyCeNN-LM",
    "tinycenn_revision": tinycenn_revision,
    "notebook": "notebooks/Laya_MemoryFusion_Colab.ipynb",
}
(export_dir / "model_meta.json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

loader_source = r'''from pathlib import Path
import torch
from huggingface_hub import snapshot_download
import laya

from tinycenn_lm.laya_lab.memory_fusion_v3 import (
    LayaMemoryFusionV3Config,
    MemoryFusionV3Attention,
)


def load_model(repo_id, device=None, token=None):
    """Load base Laya and install all accepted MemoryFusionV3 layers."""
    root = Path(
        snapshot_download(
            repo_id,
            repo_type="model",
            token=token,
            allow_patterns=[
                "adapter.pt",
                "report.json",
                "model_meta.json",
            ],
        )
    )

    payload = torch.load(
        root / "adapter.pt",
        map_location="cpu",
        weights_only=False,
    )
    if payload.get("format") != "tinycenn-laya-attention-lab-v1":
        raise RuntimeError(
            f"Unsupported TinyCeNN adapter format: {payload.get('format')!r}"
        )

    cfg = LayaMemoryFusionV3Config(**payload["lab_config"])
    agent = laya.load(cfg.model_id, device=device, token=token)
    agent.model.eval().requires_grad_(False)

    for layer_id, spec in payload["adapters"].items():
        idx = int(layer_id)
        layer = agent.model.encoder.layers[idx]
        layer_cfg = spec.get("config", {})

        replacement = MemoryFusionV3Attention(
            layer.attn,
            feature_dim=int(layer_cfg.get("feature_dim", cfg.feature_dim)),
            memory_rank=int(layer_cfg.get("memory_rank", cfg.memory_rank)),
            dilations=tuple(layer_cfg.get("dilations", cfg.dilations)),
        )

        # State dict also restores the _memory_enabled buffer.
        replacement.load_state_dict(spec["state_dict"], strict=True)
        replacement.to(agent.device)
        replacement.eval().requires_grad_(False)
        layer.attn = replacement

    agent.model.eval()
    return agent


def load_report(repo_id, token=None):
    root = Path(
        snapshot_download(
            repo_id,
            repo_type="model",
            token=token,
            allow_patterns=["report.json"],
        )
    )
    import json
    return json.loads((root / "report.json").read_text(encoding="utf-8"))
'''
(export_dir / "loader.py").write_text(loader_source, encoding="utf-8")

# Pin TinyCeNN to the exact revision used for training when available.
(export_dir / "requirements.txt").write_text(
    "git+https://github.com/NandhaKishorM/laya.git\n"
    f"git+https://github.com/vtavakkoli/TinyCeNN-LM.git@{tinycenn_revision}\n"
    "huggingface_hub>=0.34\n"
    "torch\n",
    encoding="utf-8",
)

readme = f'''---
library_name: pytorch
base_model: {cfg.model_id}
pipeline_tag: text-classification
tags:
- laya
- modernbert
- tinycenn
- memory-fusion
- linear-attention
- attention-replacement
---

# Laya MemoryFusion V3

TinyCeNN MemoryFusion V3 attention-replacement adapter for **{cfg.model_id}**.

- Status: **{report.get("status")}**
- Accepted replacement layers: **{accepted_layers}**
- Candidate layers: **{report.get("candidate_layers", [])}**
- Remaining original-attention layers: **{report.get("remaining_attention_layers", [])}**
- All attention replaced: **{report.get("all_attention_replaced", False)}**
- Feature dimension: **{cfg.feature_dim}**
- Memory rank: **{cfg.memory_rank}**
- Dilations: **{list(cfg.dilations)}**
- Recurrent GDN2 enabled by config: **{cfg.enable_recurrent_memory}**
- Fast LR: **{cfg.fast_learning_rate}**
- Core LR: **{cfg.core_learning_rate}**
- Output LR: **{cfg.output_learning_rate}**
- Measured demo speedup: **{report.get("latency", {}).get("speedup")}x**
- TinyCeNN revision: **{tinycenn_revision}**

Strict sequential acceptance was used. Rejected candidate layers were restored
to the original Laya attention before the next candidate was evaluated.

## Load

~~~python
!pip install -q -r https://huggingface.co/{HF_REPO_ID}/resolve/main/requirements.txt

from loader import load_model
agent = load_model("{HF_REPO_ID}", device="cuda")
~~~

The Hub repository stores the TinyCeNN adapter rather than duplicating the
full base Laya checkpoint. loader.py downloads the declared base model and
reconstructs each accepted MemoryFusionV3Attention layer from adapter.pt.

The saved adapter contains both full-attention and sliding-attention layer
configuration. The _memory_enabled state is restored from each layer state
dict.
'''
(export_dir / "README.md").write_text(readme, encoding="utf-8")

# Optional Colab secret: add HF_TOKEN under the key icon.
# Otherwise login() opens the Hugging Face auth flow.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    login()

api = HfApi(token=hf_token)
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    private=HF_PRIVATE,
    exist_ok=True,
)
api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=str(export_dir),
    commit_message=(
        "Upload Laya MemoryFusion V3 adapter; "
        f"accepted layers={accepted_layers}"
    ),
)

print("Uploaded:", f"https://huggingface.co/{HF_REPO_ID}")
print("Package :", export_dir)
print("Layers  :", accepted_layers)
print("Revision:", tinycenn_revision)


Uploaded: https://huggingface.co/vtava/Laya-MemoryFusion-V3
Package : /content/laya_tinycenn/memory_fusion_v3/hf_export
Layers  : [0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 17, 20, 22, 23, 24, 25, 26, 27]
Revision: 85200794015521ae38f7d628eea2774e00ca325e
